# Fine-tune FunctionGemma on Vietnamese banking tool calls

This notebook adapts Google's FunctionGemma fine-tuning workflow to this repository's generated banking function-calling dataset.

It expects the formatted JSONL dataset at `data/formatted/final_dataset.jsonl`. If you recently regenerated scenarios, run the project combine/format steps first so this notebook sees the latest rows.

## Setup

The Google guide installs PyTorch, Hugging Face Transformers, Datasets, Accelerate, Evaluate, TRL, Protobuf, and SentencePiece before loading `google/functiongemma-270m-it`. You must also accept the FunctionGemma model license on Hugging Face and authenticate before training or inference.

In [1]:
# Dependency check. Install missing packages with `uv sync` or `uv add ...` from the repo root.
import importlib.util

required_modules = [
    "torch",
    "transformers",
    "datasets",
    "accelerate",
    "evaluate",
    "trl",
    "google.protobuf",
    "sentencepiece",
    "huggingface_hub",
    "matplotlib",
]
missing_modules = []
for module_name in required_modules:
    try:
        found = importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        found = False
    if not found:
        missing_modules.append(module_name)

if missing_modules:
    print("Missing optional/runtime modules:", missing_modules)
    print("Install them from the repo root, for example: uv sync")
else:
    print("All notebook runtime dependencies are available.")

All notebook runtime dependencies are available.


In [2]:
import os
from huggingface_hub import login

# Requires a Hugging Face token with access to google/functiongemma-270m-it.
# Set HF_TOKEN in your environment before running the notebook to authenticate.
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face with HF_TOKEN.")
else:
    print(
        "HF_TOKEN is not set; model loading will be skipped unless the model is already cached and accessible."
    )

HF_TOKEN is not set; model loading will be skipped unless the model is already cached and accessible.


In [3]:
import os
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not find repository root from the current notebook path")


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "formatted" / "final_dataset.jsonl"
CHECKPOINT_DIR = (
    REPO_ROOT / "checkpoints" / "functiongemma-270m-it-banking-tool-calling"
)

BASE_MODEL = "google/functiongemma-270m-it"
HF_MODEL_REPO_NAME = "functiongemma-270m-function-calling"
HF_MODEL_REPO_ID = os.getenv("HF_MODEL_REPO_ID", HF_MODEL_REPO_NAME)
LEARNING_RATE = 5e-5
MAX_LENGTH = 1024
TEST_SIZE = 0.2
SEED = 42
PUSH_TO_HUB = False

print(f"Repository root: {REPO_ROOT}")
print(f"Dataset: {DATA_PATH}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Hugging Face repo: {HF_MODEL_REPO_ID}")

Repository root: /Users/sonphat.tran/Desktop/Project/fine-tuning-tool-calling
Dataset: /Users/sonphat.tran/Desktop/Project/fine-tuning-tool-calling/data/formatted/final_dataset.jsonl
Checkpoints: /Users/sonphat.tran/Desktop/Project/fine-tuning-tool-calling/checkpoints/functiongemma-270m-it-banking-tool-calling


## Load generated data

The repository's formatter writes FunctionGemma-style records with `messages`, `tools`, and `metadata`. Tool-response turns should already have the chat-template shape: `{"role": "tool", "content": {"name": ..., "response": ...}}`.

In [ ]:
import json
from collections import Counter
from datasets import Dataset

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Generate and format data first, for example: "
        "uv run python src/combine.py --type all && "
        "uv run python src/format.py --input data/raw/combined/final_dataset.jsonl "
        "--output data/formatted/final_dataset.jsonl"
    )

raw_rows = [
    json.loads(line)
    for line in DATA_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"Loaded {len(raw_rows):,} formatted samples")
print(
    Counter(row.get("metadata", {}).get("scenario_id", "unknown") for row in raw_rows)
)
raw_rows[0].keys()

In [5]:
def assert_functiongemma_tool_responses(row: dict) -> None:
    for message in row["messages"]:
        if message["role"] != "tool":
            continue
        content = message.get("content")
        assert isinstance(content, dict), "tool message content must be an object"
        assert set(content) >= {"name", "response"}, (
            "tool message content must include name and response"
        )


for row in raw_rows:
    assert_functiongemma_tool_responses(row)

formatted_rows = raw_rows
print(json.dumps(formatted_rows[0], ensure_ascii=False, indent=2)[:3000])

{
  "messages": [
    {
      "role": "developer",
      "content": "Ban la tro ly ngan hang thong minh, ho tro khach hang ban le tai Viet Nam.\nThong tin khach hang hien tai:\n- Ho ten: Nguyen Thi Lan\n- So dien thoai: 0908 489 398\n- Ngay hom nay: 2026-05-16\nBan co the goi cac cong cu sau de thuc hien yeu cau cua khach hang.\n"
    },
    {
      "role": "user",
      "content": "ck 1tr5 cho chị Mai bên Techcombank nha"
    },
    {
      "role": "assistant",
      "content": null,
      "tool_calls": [
        {
          "type": "function",
          "function": {
            "name": "get_beneficiary_info",
            "arguments": {}
          }
        }
      ]
    },
    {
      "role": "tool",
      "content": {
        "name": "get_beneficiary_info",
        "response": {
          "beneficiaries": [
            {
              "contact_name": "Nguyễn Thị Mai",
              "to_account": "0911222333",
              "bank_name": "TCB"
            },
            {
           

In [ ]:
import json
from collections import Counter


def encode_row_for_arrow(row: dict) -> dict:
    messages = row["messages"]
    tools = row["tools"]
    return {
        "messages": messages
        if isinstance(messages, str)
        else json.dumps(messages, ensure_ascii=False),
        "tools": tools
        if isinstance(tools, str)
        else json.dumps(tools, ensure_ascii=False),
        "metadata": row.get("metadata", {}),
    }


def decode_arrow_row(row: dict) -> dict:
    return {
        "messages": json.loads(row["messages"]),
        "tools": json.loads(row["tools"]),
        "metadata": row.get("metadata", {}),
    }


# Use raw FunctionGemma rows if available; otherwise encode current formatted_rows.
source_rows = (
    functiongemma_rows if "functiongemma_rows" in globals() else formatted_rows
)
formatted_rows = [encode_row_for_arrow(row) for row in source_rows]

print(type(formatted_rows[0]["messages"]))  # must print: <class 'str'>

dataset = Dataset.from_list(formatted_rows).train_test_split(
    test_size=TEST_SIZE,
    shuffle=True,
    seed=SEED,
)

print(dataset)
print(
    "Train scenarios:",
    Counter(row["metadata"].get("scenario_id", "unknown") for row in dataset["train"]),
)
print(
    "Test scenarios:",
    Counter(row["metadata"].get("scenario_id", "unknown") for row in dataset["test"]),
)

## Load FunctionGemma

The fine-tuning guide loads `google/functiongemma-270m-it` with `AutoModelForCausalLM` and `AutoTokenizer`, then previews the exact FunctionGemma chat template with `tokenizer.apply_chat_template(..., tools=...)`.

In [7]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model = None
tokenizer = None
MODEL_AVAILABLE = False
local_files_only = not bool(os.getenv("HF_TOKEN"))

try:
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL, local_files_only=local_files_only
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype="auto",
        device_map="auto",
        attn_implementation="eager",
        local_files_only=local_files_only,
    )
    MODEL_AVAILABLE = True
    print(f"Device: {model.device}")
    print(f"DType: {model.dtype}")
except Exception as exc:
    print(
        f"Skipping model-dependent cells because {BASE_MODEL} could not be loaded: {type(exc).__name__}: {exc}"
    )

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Device: mps:0
DType: torch.bfloat16


In [8]:
sample = decode_arrow_row(dataset["train"][0])

print("--- dataset input ---")
print(json.dumps(sample, ensure_ascii=False, indent=2)[:3000])

if tokenizer is None:
    print(
        "\nSkipping FunctionGemma formatted prompt preview because tokenizer is unavailable."
    )
else:
    formatted_prompt = tokenizer.apply_chat_template(
        sample["messages"],
        tools=sample["tools"],
        add_generation_prompt=False,
        tokenize=False,
    )
    print("\n--- FunctionGemma formatted prompt ---")
    print(formatted_prompt[:5000])

--- dataset input ---
{
  "messages": [
    {
      "role": "developer",
      "content": "Ban la tro ly ngan hang thong minh, ho tro khach hang ban le tai Viet Nam.\nThong tin khach hang hien tai:\n- Ho ten: Nguyen Thi Lan\n- So dien thoai: 0776 743 871\n- Ngay hom nay: 2026-05-19\nBan co the goi cac cong cu sau de thuc hien yeu cau cua khach hang.\n"
    },
    {
      "role": "user",
      "content": "ck 750k cho chị Hoa bên ACB nha, cảm ơn"
    },
    {
      "role": "assistant",
      "content": null,
      "tool_calls": [
        {
          "type": "function",
          "function": {
            "name": "get_beneficiary_info",
            "arguments": {}
          }
        }
      ]
    },
    {
      "role": "tool",
      "content": {
        "name": "get_beneficiary_info",
        "response": {
          "beneficiaries": [
            {
              "contact_name": "Nguyễn Thị Hoa",
              "to_account": "0908877665",
              "bank_name": "ACB"
            },
   

## Multi-turn teacher-forced evaluation

This evaluates every assistant turn in a gold multi-turn conversation. For each assistant turn, the model receives all previous gold turns and must produce the next expected assistant action. That covers clarification text, later tool calls after user clarification, chained tool calls after tool responses, and final assistant responses.

In [9]:
import re


def assistant_turn_indices(messages: list[dict]) -> list[int]:
    return [
        idx for idx, message in enumerate(messages) if message["role"] == "assistant"
    ]


def expected_action(message: dict) -> dict:
    tool_calls = message.get("tool_calls")
    if tool_calls:
        function = tool_calls[0]["function"]
        return {
            "type": "tool_call",
            "name": function["name"],
            "arguments": function.get("arguments", {}),
        }
    return {"type": "text", "content": str(message.get("content", ""))}


def clean_generated_text(text: str) -> str:
    text = re.sub(r"<start_of_turn>.*?\n", "", text)
    text = text.replace("<end_of_turn>", "")
    text = text.replace("<eos>", "")
    return text.strip()


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", clean_generated_text(text)).strip()


def split_functiongemma_args(args_text: str) -> list[str]:
    parts = []
    current = []
    in_escape = False
    i = 0
    while i < len(args_text):
        if args_text.startswith("<escape>", i):
            in_escape = not in_escape
            current.append("<escape>")
            i += len("<escape>")
            continue
        char = args_text[i]
        if char == "," and not in_escape:
            parts.append("".join(current).strip())
            current = []
        else:
            current.append(char)
        i += 1
    if current:
        parts.append("".join(current).strip())
    return [part for part in parts if part]


def parse_functiongemma_value(value: str):
    value = value.strip()
    if value.startswith("<escape>") and value.endswith("<escape>"):
        return value[len("<escape>") : -len("<escape>")]
    if value in {"true", "false"}:
        return value == "true"
    if value == "null":
        return None
    if re.fullmatch(r"-?\d+", value):
        return int(value)
    if re.fullmatch(r"-?\d+\.\d+", value):
        return float(value)
    return value.strip('"')


def parse_functiongemma_args(args_text: str) -> dict:
    args = {}
    for part in split_functiongemma_args(args_text.strip().strip("{}")):
        if ":" not in part:
            continue
        key, value = part.split(":", 1)
        args[key.strip()] = parse_functiongemma_value(value)
    return args


def parse_generated_action(output: str) -> dict:
    cleaned = clean_generated_text(output)
    call_match = re.search(r"call:([A-Za-z_][A-Za-z0-9_]*)", cleaned)
    if not call_match:
        return {"type": "text", "content": cleaned}
    remainder = cleaned[call_match.end() :]
    args_match = re.search(r"\{(.*?)\}", remainder, flags=re.DOTALL)
    return {
        "type": "tool_call",
        "name": call_match.group(1),
        "arguments": parse_functiongemma_args(args_match.group(0))
        if args_match
        else {},
        "raw": cleaned,
    }


def generate_next_assistant(
    prefix_messages: list[dict], tools: list[dict], max_new_tokens: int = 192
) -> str:
    inputs = tokenizer.apply_chat_template(
        prefix_messages,
        tools=tools,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        outputs = model.generate(
            **inputs.to(model.device),
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=max_new_tokens,
        )
    generated = outputs[0][len(inputs["input_ids"][0]) :]
    return tokenizer.decode(generated, skip_special_tokens=False)


def compare_actions(expected: dict, predicted: dict) -> dict:
    if expected["type"] != predicted["type"]:
        return {"ok": False, "type_ok": False}
    if expected["type"] == "text":
        return {
            "ok": normalize_text(expected["content"])
            == normalize_text(predicted["content"]),
            "type_ok": True,
        }
    name_ok = expected["name"] == predicted.get("name")
    args_ok = expected.get("arguments", {}) == predicted.get("arguments", {})
    return {
        "ok": name_ok and args_ok,
        "type_ok": True,
        "name_ok": name_ok,
        "args_ok": args_ok,
    }


def evaluate_assistant_turn(
    row: dict, assistant_index: int, max_new_tokens: int = 192
) -> dict:
    prefix = row["messages"][:assistant_index]
    expected = expected_action(row["messages"][assistant_index])
    output = generate_next_assistant(
        prefix, row["tools"], max_new_tokens=max_new_tokens
    )
    predicted = parse_generated_action(output)
    comparison = compare_actions(expected, predicted)
    return {
        "scenario_id": row.get("metadata", {}).get("scenario_id", "unknown"),
        "assistant_index": assistant_index,
        "expected": expected,
        "predicted": predicted,
        "output": output,
        **comparison,
    }


def evaluate_multiturn(split="test", max_samples=10, max_new_tokens=192, verbose=True):
    rows = [decode_arrow_row(row) for row in list(dataset[split])[:max_samples]]
    results = []
    for sample_index, row in enumerate(rows, start=1):
        for assistant_index in assistant_turn_indices(row["messages"]):
            result = evaluate_assistant_turn(
                row, assistant_index, max_new_tokens=max_new_tokens
            )
            result["sample_index"] = sample_index
            results.append(result)
            if verbose and not result["ok"]:
                print(
                    f"Mismatch sample={sample_index} scenario={result['scenario_id']} assistant_index={assistant_index}"
                )
                print("Expected:", result["expected"])
                print("Predicted:", result["predicted"])
                print("Raw output:", result["output"])
                print("-" * 80)
    total = len(results)
    correct = sum(result["ok"] for result in results)
    tool_results = [
        result for result in results if result["expected"]["type"] == "tool_call"
    ]
    text_results = [
        result for result in results if result["expected"]["type"] == "text"
    ]
    summary = {
        "samples": len(rows),
        "assistant_turns": total,
        "accuracy": correct / total if total else 0.0,
        "tool_call_accuracy": sum(result["ok"] for result in tool_results)
        / len(tool_results)
        if tool_results
        else None,
        "text_accuracy": sum(result["ok"] for result in text_results)
        / len(text_results)
        if text_results
        else None,
        "tool_name_accuracy": sum(
            result.get("name_ok", False) for result in tool_results
        )
        / len(tool_results)
        if tool_results
        else None,
        "tool_argument_accuracy": sum(
            result.get("args_ok", False) for result in tool_results
        )
        / len(tool_results)
        if tool_results
        else None,
    }
    print(summary)
    return summary, results

In [ ]:
# This evaluates every assistant turn, so it can be slow on CPU. Reduce max_samples if needed.
if MODEL_AVAILABLE:
    baseline_summary, baseline_results = evaluate_multiturn(
        split="test", max_samples=5, verbose=True
    )
else:
    baseline_summary, baseline_results = None, []
    print("Skipping baseline evaluation because the model is unavailable.")

## Fine-tune with TRL SFTTrainer

The default settings are conservative for the small local dataset. Increase epochs or regenerate more samples before relying on the tuned model. `RUN_TRAINING` is off by default to avoid launching a training job by accident.

In [11]:
from trl import SFTConfig, SFTTrainer

training_dataset = None
trainer = None


def render_chat_text(row: dict) -> dict:
    sample = decode_arrow_row(row)
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tools=sample["tools"],
        add_generation_prompt=False,
        tokenize=False,
    )
    return {"text": text}


if MODEL_AVAILABLE and tokenizer is not None:
    # TRL trains cleanly from plain rendered text. The encoded Dataset remains
    # available as `dataset` for metadata inspection and decoded evaluation.
    training_dataset = dataset.map(
        render_chat_text,
        remove_columns=dataset["train"].column_names,
    )

    torch_dtype = model.dtype
    sft_args = SFTConfig(
        output_dir=str(CHECKPOINT_DIR),
        max_length=MAX_LENGTH,
        packing=False,
        num_train_epochs=5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        gradient_checkpointing=False,
        optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        fp16=True if torch_dtype == torch.float16 else False,
        bf16=True if torch_dtype == torch.bfloat16 else False,
        lr_scheduler_type="constant",
        report_to="tensorboard",
        push_to_hub=PUSH_TO_HUB,
        hub_model_id=HF_MODEL_REPO_ID if PUSH_TO_HUB else None,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=training_dataset["train"],
        eval_dataset=training_dataset["test"],
        processing_class=tokenizer,
    )
else:
    print("Skipping SFTTrainer setup because the model/tokenizer is unavailable.")

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/64 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/64 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

In [12]:
RUN_TRAINING = True

if RUN_TRAINING and trainer is not None:
    trainer.train()
    trainer.save_model()
elif RUN_TRAINING:
    print("Cannot train because trainer is unavailable.")
else:
    print("Set RUN_TRAINING = True to start fine-tuning.")

Set RUN_TRAINING = True to start fine-tuning.


## Plot loss curves

In [13]:
import matplotlib.pyplot as plt

log_history = getattr(trainer.state, "log_history", []) if trainer is not None else []
train_losses = [log["loss"] for log in log_history if "loss" in log]
train_epochs = [
    log.get("epoch", idx) for idx, log in enumerate(log_history) if "loss" in log
]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
eval_epochs = [
    log.get("epoch", idx) for idx, log in enumerate(log_history) if "eval_loss" in log
]

if not train_losses and not eval_losses:
    print("No trainer logs yet. Run training first.")
else:
    if train_losses:
        plt.plot(train_epochs, train_losses, label="Training Loss")
    if eval_losses:
        plt.plot(eval_epochs, eval_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

No trainer logs yet. Run training first.


## Test after fine-tuning

After training, run the same teacher-forced multi-turn evaluation. The tuned model should more consistently match clarification text, tool names, normalized arguments, and final responses.

In [ ]:
if MODEL_AVAILABLE:
    trained_summary, trained_results = evaluate_multiturn(
        split="test", max_samples=10, verbose=True
    )
else:
    trained_summary, trained_results = None, []
    print("Skipping post-training evaluation because the model is unavailable.")

In [15]:
# Optional: save locally again, or push if PUSH_TO_HUB=True and your HF token has write access.
SAVE_FINAL_MODEL = False

if SAVE_FINAL_MODEL and trainer is not None:
    trainer.save_model(str(CHECKPOINT_DIR))
    tokenizer.save_pretrained(str(CHECKPOINT_DIR))
    print(f"Saved model and tokenizer to {CHECKPOINT_DIR}")
elif SAVE_FINAL_MODEL:
    print("Cannot save because trainer is unavailable.")
else:
    print("Set SAVE_FINAL_MODEL = True to save the current trainer/model state.")

Set SAVE_FINAL_MODEL = True to save the current trainer/model state.


## Publish To Hugging Face

This uploads the trained model to Hugging Face as `functiongemma-270m-function-calling`. Set `HF_TOKEN` in the environment before running this cell. If `HF_MODEL_REPO_ID` is unset, the notebook resolves the repo under the authenticated Hugging Face user.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, create_repo, upload_folder

PUBLISH_TO_HUB = True
PUBLISH_PRIVATE = False
PUBLISH_COMMIT_MESSAGE = "Upload FunctionGemma tool-calling fine-tune"


def latest_checkpoint_dir(base_dir: Path) -> Path | None:
    checkpoints = []
    for path in base_dir.glob("checkpoint-*"):
        try:
            step = int(path.name.rsplit("-", 1)[1])
        except (IndexError, ValueError):
            continue
        checkpoints.append((step, path))
    return max(checkpoints, default=(None, None))[1]


def has_model_artifacts(path: Path) -> bool:
    model_files = list(path.glob("*.safetensors")) + list(
        path.glob("pytorch_model*.bin")
    )
    return (path / "config.json").exists() and bool(model_files)


def resolve_hf_repo_id(api: HfApi, repo_id: str, token: str) -> str:
    if "/" in repo_id:
        return repo_id
    user = api.whoami(token=token)["name"]
    return f"{user}/{repo_id}"


def write_model_card(folder: Path, repo_id: str) -> None:
    model_card = f"""---
base_model: {BASE_MODEL}
tags:
- function-calling
- tool-calling
- functiongemma
- vietnamese
- banking
---

# {repo_id.split("/")[-1]}

Fine-tuned from `{BASE_MODEL}` on this repository's Vietnamese banking tool-calling dataset.

The model is intended for FunctionGemma-style chat templates with tool schemas passed to `tokenizer.apply_chat_template(..., tools=...)`.
"""
    (folder / "README.md").write_text(model_card, encoding="utf-8")


publish_source_dir = Path(CHECKPOINT_DIR)
if trainer is not None:
    trainer.save_model(str(CHECKPOINT_DIR))
    if tokenizer is not None:
        tokenizer.save_pretrained(str(CHECKPOINT_DIR))
elif not has_model_artifacts(publish_source_dir):
    latest_checkpoint = latest_checkpoint_dir(Path(CHECKPOINT_DIR))
    if latest_checkpoint is not None:
        publish_source_dir = latest_checkpoint

if not PUBLISH_TO_HUB:
    print("Set PUBLISH_TO_HUB = True to upload the model to Hugging Face.")
elif not hf_token:
    raise RuntimeError("HF_TOKEN must be set before publishing to Hugging Face.")
elif not has_model_artifacts(publish_source_dir):
    raise RuntimeError(
        f"No model artifacts found in {publish_source_dir}. Train or save a model first."
    )
else:
    api = HfApi(token=hf_token)
    resolved_repo_id = resolve_hf_repo_id(api, HF_MODEL_REPO_ID, hf_token)
    create_repo(
        repo_id=resolved_repo_id,
        token=hf_token,
        private=PUBLISH_PRIVATE,
        exist_ok=True,
    )
    write_model_card(publish_source_dir, resolved_repo_id)
    upload_folder(
        repo_id=resolved_repo_id,
        folder_path=str(publish_source_dir),
        token=hf_token,
        commit_message=PUBLISH_COMMIT_MESSAGE,
        ignore_patterns=[
            "runs/*",
            "checkpoint-*",
            "optimizer.pt",
            "scheduler.pt",
            "rng_state.pth",
            "trainer_state.json",
            "training_args.bin",
        ],
    )
    print(f"Published model to https://huggingface.co/{resolved_repo_id}")